# 21 — Learning from Data

**Master syllabus coverage:** Added foundation and V2 3.10

## Why this matters

Before deep networks, a learner needs the entire empirical workflow: define a task, split data correctly, fit a baseline, choose metrics, and recognize leakage.

## Learning objectives

- Assign distinct roles to train, validation, and test splits.
- Train linear regression with mean-squared error.
- Train logistic regression from logits with binary cross-entropy.
- Identify feature, preprocessing, duplicate, and temporal leakage.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Supervised learning separates fitting from evaluation

A dataset contains examples from an unknown process. The training split changes model
parameters. The validation split selects hyperparameters and stopping decisions. The
test split estimates final generalization once. Reusing test feedback turns it into a
validation set and biases the estimate.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

rng = np.random.default_rng(42)
n = 300
x = rng.normal(size=(n, 2)).astype(np.float32)
y_reg = (2.0 * x[:, 0] - 3.0 * x[:, 1] + 0.5 + rng.normal(scale=0.3, size=n)).astype(np.float32)
y_cls = (y_reg > np.median(y_reg)).astype(np.int64)

indices = rng.permutation(n)
train_idx, val_idx, test_idx = indices[:200], indices[200:250], indices[250:]
assert not (set(train_idx) & set(val_idx) or set(train_idx) & set(test_idx))
print("split sizes:", len(train_idx), len(val_idx), len(test_idx))


## 2. Linear regression

The model $\hat y=Xw+b$ minimizes mean squared error
$\frac1N\sum_i(\hat y_i-y_i)^2$. The closed-form least-squares solution is useful for
understanding; gradient methods scale to models where no closed form exists.


In [ ]:
X_train = torch.tensor(x[train_idx])
y_train = torch.tensor(y_reg[train_idx])
X_val = torch.tensor(x[val_idx])
y_val = torch.tensor(y_reg[val_idx])

regressor = torch.nn.Linear(2, 1)
optimizer = torch.optim.SGD(regressor.parameters(), lr=0.05)
for _ in range(200):
    prediction = regressor(X_train).squeeze(-1)
    loss = F.mse_loss(prediction, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    val_mse = F.mse_loss(regressor(X_val).squeeze(-1), y_val)
print("learned weight:", regressor.weight.detach().numpy().round(2),
      "bias:", regressor.bias.item(), "val MSE:", val_mse.item())
assert val_mse < 0.2


## 3. Logistic regression

Binary logistic regression emits logit $z=x\cdot w+b$. Sigmoid maps it to
$p(y=1\mid x)$. Binary cross-entropy fits probabilities; thresholding at 0.5 is a later
decision rule and may be inappropriate under class imbalance or asymmetric costs.


In [ ]:
classifier = torch.nn.Linear(2, 1)
targets = torch.tensor(y_cls[train_idx], dtype=torch.float32)
optimizer = torch.optim.SGD(classifier.parameters(), lr=0.1)
for _ in range(250):
    logits = classifier(X_train).squeeze(-1)
    loss = F.binary_cross_entropy_with_logits(logits, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    val_logits = classifier(X_val).squeeze(-1)
    val_predictions = (val_logits.sigmoid() >= 0.5).long()
    accuracy = (val_predictions == torch.tensor(y_cls[val_idx])).float().mean()
print("validation accuracy:", accuracy.item())
assert accuracy > 0.85


## 4. Leakage creates performance you cannot deploy

Leakage occurs when training inputs or preprocessing contain information unavailable at
prediction time. Common examples: normalizing with all splits, near-duplicate documents
across splits, future information in time-series features, and packing adjacent slices
of the same source into different splits.


In [ ]:
# A deliberately leaked feature: it directly contains the label.
clean_features = torch.tensor(x[test_idx])
test_targets = torch.tensor(y_cls[test_idx])
leaked_features = torch.column_stack((clean_features, test_targets.float()))
leaked_rule = (leaked_features[:, -1] > 0.5).long()
leaked_accuracy = (leaked_rule == test_targets).float().mean()
print("impossible-looking leaked accuracy:", leaked_accuracy.item())
assert leaked_accuracy == 1.0

# Fit preprocessing statistics only on training data.
mean = X_train.mean(dim=0, keepdim=True)
std = X_train.std(dim=0, keepdim=True).clamp_min(1e-6)
normalized_test = (clean_features - mean) / std
assert normalized_test.shape == clean_features.shape


## Failure modes to recognize

- **Test-set tuning:** reported test performance is optimistically biased.
- **Preprocessing leakage:** validation/test statistics influence training transforms.
- **Accuracy-only evaluation:** class imbalance hides a useless classifier.
- **No baseline:** complexity is added without proving a simpler model is insufficient.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Derive the linear-regression gradients for weight and bias and implement them without autograd.
2. Create an imbalanced classification dataset and compare accuracy, precision, recall, and a confusion matrix.
3. List three ways document-level leakage could enter an SLM dataset and how to prevent each.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can build and evaluate a baseline with leakage-safe splits and explain what information influenced every reported metric.

**Next:** 22 — Neurons, activations, and MLPs.
